# 05 - Multispectral RGB & Custom Composite Builder

**What:** Build Sentinel-2 true-color and false-color RGB imagery from a clear median
`COPERNICUS/S2_SR_HARMONIZED` image over an AOI, render preset composites inline as
thumbnails, build a custom 3-band composite, and export a small RGB GeoTIFF download URL.

**Why:** Optical core feature of the RAVI plugin. Ports the legacy QGIS dialog logic into a
standalone, GEE-authenticated notebook so the render modes and visualization parameters can be
validated independently of QGIS.

**Legacy reference** (`git show legacy:ravi_dialog.py`):
- `load_rgb` (~L2604-2712) — selects bands `B1..B12,B8A`, clips to AOI, downloads GeoTIFF
  (`scale=10`, `format=GeoTIFF`, `crs=EPSG:4326`); default scaling `min_val=200, max_val=2300`.
- `rgb_rendering` (~L2727-2753) — maps `QComboBox_composition` text to the per-mode band triplet.
- `QComboBox_composition.addItems(...)` (~L407) — the exact preset mode names.
- `composite_clicked` (~L2899) — custom composite picking any of B1-B12/B8A for R/G/B.
- Collection build (~L3628) — `S2_SR_HARMONIZED.filterDate().filterBounds()` + cloud filter.

> Requires a GEE-authenticated environment (`ee.Initialize`). Thumbnails are fetched over HTTP
> from the Earth Engine thumbnail endpoint and shown inline.

In [ ]:
# --- Setup ---
import os
import ee
import requests
from IPython.display import Image, display

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)

# Optional interactive map (guarded - not required for the notebook to run).
try:
    import geemap
    HAS_GEEMAP = True
except Exception:
    HAS_GEEMAP = False
    print("geemap not available - inline thumbnails will be used instead.")

print("Earth Engine initialized.")

In [ ]:
# --- AOI (from project shapefile, same area as dev/testing.ipynb) + dates + clear median image ---
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip").geometry()

START_DATE = "2024-06-01"
END_DATE = "2024-09-30"
CLOUD_PCT = 20  # max CLOUDY_PIXEL_PERCENTAGE per scene (legacy 'nuvem' filter)

# Mirrors the legacy collection build (~L3628): S2_SR_HARMONIZED, date + bounds, then
# a scene-level cloud-percentage filter. A median reduces to a single clear image.
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_PCT))
)

print("Scenes in filtered collection:", collection.size().getInfo())

# Legacy load_rgb selects this exact band set before clipping.
BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]

image = collection.select(BANDS).median().clip(aoi)
print("Median image bands:", image.bandNames().getInfo())

In [ ]:
# --- RENDER_MODES: exact band triplets from legacy rgb_rendering (~L2727) ---
# Legacy returns Sentinel-2 band NUMBERS; here mapped to the equivalent band NAMES
# (R, G, B) used for GEE visualization. Keys are the exact QComboBox_composition labels.
RENDER_MODES = {
    "Real color (RGB)": ["B4", "B3", "B2"],   # Red, Green, Blue
    "Red-NIR-Green":    ["B4", "B8", "B3"],   # Red, NIR, Green
    "NIR-Red-Green":    ["B8", "B4", "B3"],   # NIR, Red, Green
    "SWIR2-NIR-Green":  ["B12", "B8", "B3"],  # SWIR2, NIR, Green
    "SWIR1-NIR-SWIR2":  ["B11", "B8", "B12"], # SWIR1, NIR, SWIR2
}

# Legacy load_rgb default scaling (min_val=200, max_val=2300) for surface reflectance.
DEFAULT_MIN = 200
DEFAULT_MAX = 2300

for name, bands in RENDER_MODES.items():
    print(f"{name:18s} -> R={bands[0]}, G={bands[1]}, B={bands[2]}")

In [ ]:
# --- Build an RGB visualization thumbnail for a given band triplet + min/max ---
def rgb_thumb_url(img, bands, region, min_val=DEFAULT_MIN, max_val=DEFAULT_MAX, dimensions=512):
    """Return a getThumbURL for a 3-band RGB visualization.

    bands : [R, G, B] band names
    min_val/max_val : linear contrast stretch (auto min/max scaling, like the legacy
                      StretchToMinimumMaximum contrast enhancement on the loaded raster).
    """
    vis = {
        "bands": bands,
        "min": min_val,
        "max": max_val,
        "region": region,
        "dimensions": dimensions,
        "format": "png",
    }
    return img.getThumbURL(vis)


def show_rgb(img, bands, region, title="", min_val=DEFAULT_MIN, max_val=DEFAULT_MAX):
    """Fetch and display an RGB thumbnail inline."""
    url = rgb_thumb_url(img, bands, region, min_val, max_val)
    if title:
        print(f"{title}  (R={bands[0]} G={bands[1]} B={bands[2]}, min={min_val} max={max_val})")
    print(url)
    resp = requests.get(url)
    resp.raise_for_status()
    display(Image(data=resp.content))
    return url


# Region for thumbnails (AOI bounds as GeoJSON coordinates).
region = aoi.bounds().getInfo()["coordinates"]

In [ ]:
# --- Render every preset mode inline ---
for name, bands in RENDER_MODES.items():
    show_rgb(image, bands, region, title=name)

In [ ]:
# --- Custom composite builder ---
# Pick ANY three bands from the available set for the R / G / B channels.
# Available bands: B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12
custom_R = "B11"  # e.g. SWIR1
custom_G = "B12"  # e.g. SWIR2
custom_B = "B4"   # e.g. Red

custom_bands = [custom_R, custom_G, custom_B]
available = image.bandNames().getInfo()
missing = [b for b in custom_bands if b not in available]
assert not missing, f"Unknown band(s): {missing}. Choose from {available}"

# You can also override the stretch for the custom composite.
show_rgb(image, custom_bands, region, title="Custom composite",
         min_val=DEFAULT_MIN, max_val=DEFAULT_MAX)

In [ ]:
# --- Export / download an RGB GeoTIFF (small region) ---
# Mirrors legacy load_rgb getDownloadURL params: scale=10, GeoTIFF, EPSG:4326.
# Pick a mode (or use custom_bands) and emit a download URL - the file is NOT fetched here.
export_bands = RENDER_MODES["Real color (RGB)"]

download_url = image.select(export_bands).getDownloadURL(
    {
        "scale": 10,
        "region": region,
        "format": "GEO_TIFF",
        "crs": "EPSG:4326",
    }
)
print("RGB GeoTIFF download URL (bands", export_bands, "):")
print(download_url)

# Optional: preview on an interactive map if geemap is installed.
if HAS_GEEMAP:
    m = geemap.Map()
    m.centerObject(aoi, 13)
    m.addLayer(image, {"bands": export_bands, "min": DEFAULT_MIN, "max": DEFAULT_MAX}, "Real color")
    display(m)